In [1]:
import sys
import os
import torch
from pathlib import Path

def get_project_info() -> tuple[Path, Path]:
    current = Path.cwd().resolve()
    root = current
    for parent in [current, *current.parents]:
        if (parent / "toy_transformers").exists():
            root = parent
            break
    return root, current

if 'ROOT_DIR' not in globals():
    ROOT_DIR, EXPERIMENT_DIR = get_project_info()
    if str(ROOT_DIR) not in sys.path:
        sys.path.append(str(ROOT_DIR))
    if Path.cwd() != ROOT_DIR:
        os.chdir(ROOT_DIR)

from toy_transformers.models import gptv4
from toy_transformers import tokenization
from toy_transformers.model_io import load_model
from toy_transformers.data import S3Sync
from toy_transformers.config import TrainingConfig

print(f"ROOT_DIR: {ROOT_DIR}")

ROOT_DIR: /Users/sriman/dev/apps/toy-transformers


In [2]:
RUN_NAME = "baseline-fineweb-edu"
# CHECKPOINT_STEP = 37500
DEVICE = "mps"  # change to 'cuda' or 'cpu' as needed

# Load bucket name from .env
env_path = ROOT_DIR / ".env"
BUCKET = None
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line.startswith("BUCKET="):
            BUCKET = line.split("=", 1)[1].strip()
            break

if BUCKET is None:
    raise ValueError("BUCKET not found in .env")

sync = S3Sync(remote_base=f"s3://{BUCKET}/toy-transformers", local_root=ROOT_DIR)

In [3]:
# Load the training config to get tokenizer path and model config
cfg = TrainingConfig.from_json(ROOT_DIR / "configs" / f"{RUN_NAME}.json")

# Pull tokenizer from S3 if not present
print("Pulling tokenizer...")
sync.pull_atomic(cfg.tokenizer.path)
cfg.tokenizer.load(ROOT_DIR)
print(f"Vocab size: {cfg.tokenizer.vocab_size}, BOS={cfg.tokenizer.bos_id}, PAD={cfg.tokenizer.pad_id}")

# Pull checkpoint from S3 if not present
CKPT_REL = f"runs/{RUN_NAME}/checkpoints/final"
CKPT_DIR = ROOT_DIR / CKPT_REL
print(f"Pulling checkpoint: {CKPT_REL}")
sync.pull(CKPT_REL)
print(f"Checkpoint local path: {CKPT_DIR}")

Pulling tokenizer...
Vocab size: 32768, BOS=0, PAD=1
Pulling checkpoint: runs/baseline-fineweb-edu/checkpoints/final
Checkpoint local path: /Users/sriman/dev/apps/toy-transformers/runs/baseline-fineweb-edu/checkpoints/final


In [4]:
torch.set_float32_matmul_precision("medium")

model = cfg.model.build_model(vocab_size=cfg.tokenizer.vocab_size, device=DEVICE)
print(f"{model.get_num_parameters(as_str=True)} parameters")

load_model(CKPT_DIR, cfg, model, device=DEVICE)
model.eval()
print("Checkpoint loaded successfully")

100.682m parameters
Checkpoint loaded successfully


In [5]:
vocab = tokenization.Vocabulary.load(ROOT_DIR / cfg.tokenizer.path)

def encode(text: str) -> torch.Tensor:
    ids = [cfg.tokenizer.bos_id] + vocab.encode(text.encode("utf-8"))
    return torch.tensor([ids], dtype=torch.long, device=DEVICE)

def decode_tokens(token_ids: list[int]) -> str:
    byte_chunks = vocab.decode(token_ids)
    return b"".join(byte_chunks).decode("utf-8", errors="replace")

In [ ]:
prompt = "<BOS>The best way to heat up a pizza"
MAX_NEW_TOKENS = 300
TEMPERATURE = 1.0
TOPK = 40

idx = encode(prompt)
print(prompt, end="", flush=True)

for token in model.generate(idx, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, topk=TOPK):
    if token.item() == cfg.tokenizer.bos_id:
        break
    print(decode_tokens([token.item()]), end="", flush=True)

print()

<BOS>The best way to heat up a pizza or a ham sandwich is to soak it. If you find puffs of pizza or something made with marinara, soak some in hot sauce like lemon, cilantro or oregano till it becomes mushy.
When I was little back in high school, I had seen cheesemakers in the basement playing with one of my favorite hot dogs, the Mummy. This is my favorite hot dog. It was the first time I saw a Mummy! There are many great things to do on a hot dog, both in terms of preparation and fun.
The idea is to make a hot dog fun. Let's take the fun one step further. Take the food out of the hot dog (it's hard to be a great hot dog to cook one), add a little pepper and cilantro, and put in the meat or gravy and give it a bit more texture for it.
After an hour, your hot dog should be pretty blubber but not crumpled. If you find some pepper on the inside, it will help make your hot dogs easier to put in the hot dog. Just make sure to add more gravy or gravy to make it more appetizing and enjoyable

In [7]:
from toy_transformers.eval import eval as hellaswag_eval

eval_parquet = ROOT_DIR / "data/evals/hellaswag/hellaswag.parquet"
block_size = cfg.model.config["block_size"]

results = hellaswag_eval(
    model, vocab, eval_parquet, block_size, device=DEVICE, batch_size=4,
    bos_id=cfg.tokenizer.bos_id, pad_id=cfg.tokenizer.pad_id, limit=300
)

print(f"acc:      {results['acc']:.4f}")
print(f"acc_norm: {results['acc_norm']:.4f}")
print(f"n:        {results['num_total']}")

[EVALUATE] 300 examples, batch_size=4


evaluating:   7%|▋         | 5/75 [00:01<00:19,  3.62it/s]


KeyboardInterrupt: 